In [0]:
from pyspark.sql.functions import expr, current_timestamp

### Leitura do CSV ICAO/IATA

In [0]:
df_aero_siglas = spark.read \
    .option("header", True) \
    .option("delimiter", ",") \
    .csv("/Volumes/ct-esteira-dados-aviacao/raw/vl_aero_icao_iata/*.csv")

# padroniza o nome das colunas para minusculas.

df_aero_siglas = df_aero_siglas.toDF(*[c.strip().lower() for c in df_aero_siglas.columns])


#add coluna com timestamp para controle de carga.

df_aero_siglas = df_aero_siglas.withColumn("load_data", expr("current_timestamp() - interval 3 hours"))
display(df_aero_siglas)


In [0]:
df_aero_siglas \
    .write \
    .format('delta') \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("codigo_icao") \
    .saveAsTable("`ct-esteira-dados-aviacao`.raw.dim_aero_siglas")
